In [ ]:
import os

# ============================================================
# CONFIGURAÇÃO — adapte para o seu domínio
# ============================================================

# Descrição do domínio que aparecerá no prompt do LLM
DOMAIN_DESCRIPTION = os.environ.get(
    'DOMAIN_DESCRIPTION',
    'transcricoes de chamadas operacionais'  # descreva o seu domínio aqui
)

# Categoria catch-all (mutuamente exclusiva com todas as outras)
CATCHALL_CATEGORY = os.environ.get('CATCHALL_CATEGORY', 'Outros assuntos')

# ── Defina as categorias e suas palavras-chave/contexto ──────────────────────
# Substitua pelas categorias do SEU domínio.
# Quanto mais específicas as palavras-chave, melhor a classificação.
VALID_CATEGORIES = [
    # Exemplo genérico — substitua pelos nomes das suas categorias:
    'Categoria A',
    'Categoria B',
    'Categoria C',
    CATCHALL_CATEGORY,
]

CATEGORY_DESCRIPTIONS = {
    'Categoria A': {
        'keywords': ['palavra-chave-1', 'palavra-chave-2', 'frase típica'],
        'context': 'Descreva quando esta categoria deve ser aplicada.'
    },
    'Categoria B': {
        'keywords': ['palavra-chave-3', 'palavra-chave-4'],
        'context': 'Descreva quando esta categoria deve ser aplicada.'
    },
    'Categoria C': {
        'keywords': ['palavra-chave-5'],
        'context': 'Descreva quando esta categoria deve ser aplicada.'
    },
    CATCHALL_CATEGORY: {
        'keywords': ['saudacoes', 'teste de audio', 'conversa informal'],
        'context': 'Conversas que não se encaixam em nenhuma categoria operacional acima.'
    },
}
# ============================================================


def build_prompt(categories: list, descriptions: dict, domain: str, catchall: str) -> str:
    """Constrói o prompt de classificação dinamicamente a partir das categorias configuradas."""
    category_section = ''
    for cat in categories:
        desc = descriptions.get(cat, {})
        keywords = ', '.join(f'"{k}"' for k in desc.get('keywords', []))
        context = desc.get('context', '')
        category_section += f'\n-   **{cat}:**\n'
        if keywords:
            category_section += f'    -   **Palavras-chave:** {keywords}\n'
        if context:
            category_section += f'    -   **Contexto:** {context}\n'

    categories_json = str(categories).replace("'", '"')

    return f"""Você é um classificador especializado em {domain}.
Sua tarefa é analisar o conteúdo textual e atribuir as categorias mais relevantes.

### Regras de Classificação
1. **Precisão:** Somente atribua uma categoria se houver evidência clara e explícita no texto.
2. **Exclusividade do catch-all:** A categoria ["{catchall}"] é mutuamente exclusiva com todas as outras.
   Use-a somente quando nenhuma outra categoria se aplicar com certeza.
3. **Múltiplas categorias:** É permitido retornar múltiplas categorias se o texto cobrir mais de um tema.
4. **Formato de saída:** Retorne apenas uma lista JSON válida (ex: ["Categoria A", "Categoria B"]).
5. **Sem invenção:** Use apenas categorias da lista: {categories_json}.

### Categorias Válidas e Critérios
{category_section}

### Processe a transcrição abaixo:
"""


PROMPT_OTIMIZADO = build_prompt(
    VALID_CATEGORIES,
    CATEGORY_DESCRIPTIONS,
    DOMAIN_DESCRIPTION,
    catchall=CATCHALL_CATEGORY,
)

print('Prompt construído com sucesso.')
print(f'Categorias configuradas: {VALID_CATEGORIES}')

In [ ]:
import pandas as pd

VALIDATION_CSV = os.environ.get('VALIDATION_CSV', '../truths/sample_data.csv')
TRANSCRIPTION_COLUMN = 'transcription'
LABEL_COLUMN         = 'Assunto tratado'

df_validation = pd.read_csv(VALIDATION_CSV)
df_validation

In [ ]:
import pandas as pd
import re
import unicodedata
import ast
import json

def normalizar_texto(texto: str) -> str:
    if not isinstance(texto, str) or pd.isna(texto):
        return ''
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join([c for c in texto if unicodedata.category(c) != 'Mn'])
    texto = re.sub(r'[^\w\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    texto = ' '.join([word for word in texto.split() if len(word) > 1])
    return texto

def normalizar_assunto_tratado(texto):
    if not isinstance(texto, str):
        return []
    try:
        data = json.loads(texto)
        if isinstance(data, dict) and 'choices' in data:
            return data['choices']
    except (json.JSONDecodeError, TypeError):
        pass
    try:
        data = ast.literal_eval(texto)
        if isinstance(data, list):
            return data
    except (ValueError, SyntaxError):
        pass
    return [texto.strip()]

df_validation[TRANSCRIPTION_COLUMN] = df_validation[TRANSCRIPTION_COLUMN].apply(normalizar_texto)
df_validation[LABEL_COLUMN] = df_validation[LABEL_COLUMN].apply(normalizar_assunto_tratado)
df_validation

In [ ]:
import os
from llama_index.core.llms import ChatMessage

# ============================================================
# Escolha o provider LLM: 'bedrock' | 'openai' | 'anthropic'
# Certifique-se de ter instalado o extra correspondente:
#   uv pip install 'subject-classification[bedrock]'
#   uv pip install 'subject-classification[openai]'
#   uv pip install 'subject-classification[anthropic]'
# ============================================================
LLM_PROVIDER = os.environ.get('LLM_PROVIDER', 'bedrock')
LLM_MODEL    = os.environ.get('LLM_MODEL', '')

if LLM_PROVIDER == 'bedrock':
    from llama_index.llms.bedrock import Bedrock
    llm = Bedrock(
        model=LLM_MODEL or 'anthropic.claude-3-5-sonnet-20241022-v2:0',
        context_size=200000,
        max_tokens=4096,
    )
elif LLM_PROVIDER == 'openai':
    from llama_index.llms.openai import OpenAI
    llm = OpenAI(
        model=LLM_MODEL or 'gpt-4o',
        api_key=os.environ.get('OPENAI_API_KEY'),
    )
elif LLM_PROVIDER == 'anthropic':
    from llama_index.llms.anthropic import Anthropic
    llm = Anthropic(
        model=LLM_MODEL or 'claude-sonnet-4-6',
        api_key=os.environ.get('ANTHROPIC_API_KEY'),
    )
else:
    raise ValueError(f"Provider '{LLM_PROVIDER}' não suportado. Use: bedrock, openai ou anthropic.")


def get_audio_subject(transcription: str) -> str:
    try:
        messages = [
            ChatMessage(role='system', content=PROMPT_OTIMIZADO),
            ChatMessage(role='user', content=str(transcription)),
        ]
        return llm.chat(messages).message.content
    except Exception as e:
        return f'Erro ao classificar: {str(e)}'

In [ ]:
import numpy as np
import ast

def pipe(transcription):
    subject_list = get_audio_subject(transcription)
    print(f"Retorno LLM: {subject_list}")

    #tentar converter a string para lista
    try: 
        subject_list = ast.literal_eval(subject_list) 
    except Exception: 
        subject_list = [subject_list] # fallback, se vier texto solto

    return subject_list


In [ ]:
import warnings
warnings.filterwarnings('ignore')

preds = []

for idx, row in df_validation.iterrows():
    print(f"\nProcessando áudio {idx}...\n")
    print(f"transcrição: {row['transcription']}")
    print(f"Resultado esperado: {row['Assunto tratado']}")
    subj = pipe(row['transcription'])
    preds.append(subj)

df_results = df_validation.copy()

df_results["B"] = preds

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score

def ensure_list(val):
    if isinstance(val, list):
        return [str(x).strip() for x in val]
    if pd.isna(val):
        return []
    return [str(val).strip()]

# garantir listas
df_results["Assunto tratado"] = df_results["Assunto tratado"].apply(ensure_list)
df_results["B"]   = df_results["B"].apply(ensure_list)

# binarizar
mlb = MultiLabelBinarizer()
y_true = mlb.fit_transform(df_results["Assunto tratado"])
y_pred = mlb.transform(df_results["B"])

# métricas
print("Subset Accuracy:", accuracy_score(y_true, y_pred))
print("F1 micro:", f1_score(y_true, y_pred, average="micro"))
print("F1 macro:", f1_score(y_true, y_pred, average="macro"))
print("F1 weighted:", f1_score(y_true, y_pred, average="weighted"))
print("F1 samples:", f1_score(y_true, y_pred, average="samples"))
print("Relatório de classificação:")
print(classification_report(y_true, y_pred, target_names=mlb.classes_))